<a href="https://colab.research.google.com/github/thanhuy8888/-n_Systematic-Review-AI/blob/main/experiments/colab_finetune_gpu.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Fine-tune & tối ưu screening model trên Google Colab (GPU T4)

Notebook duy nhất cho toàn bộ thí nghiệm GPU của đồ án — nhanh hơn CPU local ~16 lần.

**Trước khi chạy:** `Runtime` → `Change runtime type` → chọn **T4 GPU** → Save. Chạy cell Setup trước, sau đó chọn phần cần làm:

| Phần | Nội dung | Thời gian | Trạng thái |
|---|---|---|---|
| **1A** | distil-biobert 12ep freeze2 | ~10 phút | ✅ Đã chạy 11/06/2026 — kết quả là model chính thức (ROC-AUC 87.2%, recall 79.0%) |
| **1B** | PubMedBERT đầy đủ 12 layers | ~30 phút | ✅ Đã chạy 11/06/2026 — thua distil 6/7 chỉ số (kết luận ablation) |
| **2** | Sweep 4 cấu hình + ổn định 3 seeds | ~40–50 phút | ✅ Đã chạy — ROC-AUC 87.2%, recall 79.0%, WSS@95 33.4% |
| **3** | Focal Loss + 5 seeds + Ensemble | ~60–75 phút | ⏳ Bước tiếp theo |

Phần 1 & 2 giữ lại để tái lập kết quả; muốn cải thiện recall thì sau Setup nhảy thẳng xuống **Phần 3**.

## Setup (bắt buộc, chạy đầu tiên)

In [7]:
# Kiểm tra GPU (phải thấy Tesla T4) + lấy code + cài thư viện
!nvidia-smi -L
!git clone https://github.com/thanhuy8888/-n_Systematic-Review-AI.git srai
%cd srai
!pip install -q transformers seaborn

GPU 0: Tesla T4 (UUID: GPU-35491363-d3c6-1c3e-5209-af5a1c0ac011)
Cloning into 'srai'...
remote: Enumerating objects: 379, done.
remote: Counting objects: 100% (379/379), done.
remote: Compressing objects: 100% (315/315), done.
remote: Total 379 (delta 137), reused 201 (delta 52), pack-reused 0 (from 0)
Receiving objects: 100% (379/379), 22.76 MiB | 8.18 MiB/s, done.
Resolving deltas: 100% (137/137), done.
Updating files: 100% (115/115), done.
/content/srai/srai


In [8]:
%cd /content/srai
!git pull origin main

/content/srai
remote: Enumerating objects: 7, done.
remote: Counting objects: 100% (7/7), done.
remote: Compressing objects: 100% (2/2), done.
remote: Total 4 (delta 2), reused 4 (delta 2), pack-reused 0 (from 0)
Unpacking objects: 100% (4/4), 1.51 KiB | 221.00 KiB/s, done.
From https://github.com/thanhuy8888/-n_Systematic-Review-AI
 * branch            main       -> FETCH_HEAD
   8d6b9bc..8a8d2e8  main       -> origin/main
Updating 8d6b9bc..8a8d2e8
Fast-forward
 experiments/colab_finetune_gpu.ipynb | 4 ++--
 1 file changed, 2 insertions(+), 2 deletions(-)


---
# PHẦN 1 — Hai thí nghiệm cơ bản (đã chạy, giữ để tái lập)

## 1A — distil-biobert, 12 epochs, freeze 2

Công thức thắng của vòng 1: script tự lưu checkpoint epoch có val ROC-AUC cao nhất (lần chạy 11/06 đỉnh rơi ở epoch 5) và dùng nó cho đánh giá cuối.

In [ ]:
!python experiments/baselines/finetune_pubmedbert.py \
    --model nlpie/distil-biobert --epochs 12 --lr 3e-5 \
    --batch-size 16 --max-len 256 --freeze-layers 2

In [ ]:
# Đóng gói kết quả 1A để tải về
!zip -r -q ket_qua_A_distil_12ep.zip \
    sr_core/screening_model/finetuned_pubmedbert \
    sr_core/screening_model/finetuned_meta.json \
    experiments/results/transformer_confusion_matrix.png \
    experiments/results/transformer_roc_curve.png \
    experiments/results/transformer_pr_curve.png \
    experiments/results/transformer_probability_distribution.png
from google.colab import files
files.download('ket_qua_A_distil_12ep.zip')

## 1B — PubMedBERT bản đầy đủ (12 layers, freeze 6)

Kết quả 11/06: thua bản distil ở 6/7 chỉ số — bằng chứng "model to không thắng với dataset 8K mẫu" cho chương đánh giá.

⚠️ Cell này GHI ĐÈ kết quả 1A trong thư mục — chạy cell tải về của 1A trước.

In [ ]:
!python experiments/baselines/finetune_pubmedbert.py \
    --epochs 8 --lr 2e-5 \
    --batch-size 16 --max-len 256 --freeze-layers 6

In [ ]:
# Đóng gói kết quả 1B để tải về
!zip -r -q ket_qua_B_pubmedbert_full.zip \
    sr_core/screening_model/finetuned_pubmedbert \
    sr_core/screening_model/finetuned_meta.json \
    experiments/results/transformer_confusion_matrix.png \
    experiments/results/transformer_roc_curve.png \
    experiments/results/transformer_pr_curve.png \
    experiments/results/transformer_probability_distribution.png
from google.colab import files
files.download('ket_qua_B_pubmedbert_full.zip')

---
# PHẦN 2 — Sweep cấu hình + kiểm tra ổn định (bước tiếp theo)

Tìm cấu hình tốt nhất một cách **có kỷ luật, không rò rỉ dữ liệu**:

1. **Sweep 4 cấu hình** (~30 phút) quanh vùng đã biết là tốt. Cấu hình thắng được chọn theo **validation ROC-AUC** — tuyệt đối không chọn theo test (test-set fishing = gian lận số liệu).
2. **Kiểm tra ổn định** (~12 phút): train lại config thắng với 3 seed khởi tạo (tập test giữ nguyên nhờ `--init-seed`) → số **mean ± std** thuyết phục khi bảo vệ.
3. **Đóng gói** model thắng + bảng tổng hợp để tải về deploy.

**Đã rút gọn để chạy nhanh:** từ vòng 1 ta biết model 12-layer đầy đủ thua bản distil 6-layer (thí nghiệm 1B) → bỏ config BioLinkBERT; bỏ luôn `max-len 320` và `batch 32` (tốn thời gian, lực bẩy nhỏ). Chỉ còn 4 config distil quanh điểm tốt nhất.

**Có lưu cache:** mỗi config xong được lưu vào `sweep_results/`; nếu Colab rớt giữa chừng, chạy lại cell sẽ **bỏ qua các config đã xong** (in `[cache] ... bỏ qua`) thay vì train lại từ đầu.

Config `distil_lr3e5_fz2` chính là thí nghiệm 1A vô địch vòng 1 — làm mốc chuẩn (chạy lại 1 lần để lấy số validation cho việc xếp hạng).

In [ ]:
import subprocess, json, shutil, os
import pandas as pd

CONFIGS = {
    #  tên                (model,                  epochs, lr,     batch, maxlen, freeze)
    "distil_lr3e5_fz2":  ("nlpie/distil-biobert",  12, "3e-5", 16, 256, 2),  # mốc chuẩn = 1A
    "distil_lr2e5_fz2":  ("nlpie/distil-biobert",  12, "2e-5", 16, 256, 2),
    "distil_lr5e5_fz2":  ("nlpie/distil-biobert",  12, "5e-5", 16, 256, 2),
    "distil_lr3e5_fz1":  ("nlpie/distil-biobert",  12, "3e-5", 16, 256, 1),
}

META = "sr_core/screening_model/finetuned_meta.json"
CKPT_INFO = "sr_core/screening_model/finetuned_pubmedbert_ckpt/ckpt_info.json"
MODEL_DIR = "sr_core/screening_model/finetuned_pubmedbert"
CHARTS = ["transformer_confusion_matrix.png", "transformer_roc_curve.png",
          "transformer_pr_curve.png", "transformer_probability_distribution.png"]

os.makedirs("sweep_results", exist_ok=True)
rows = []
best_val = -1.0
for name, (model, ep, lr, bs, ml, fz) in CONFIGS.items():
    print(f"\n{'='*20} {name} {'='*20}")
    dest = f"sweep_results/{name}"
    cached_meta = os.path.join(dest, "finetuned_meta.json")
    cached_info = os.path.join(dest, "ckpt_info.json")
    cached_model = os.path.join(dest, "model")
    if os.path.exists(cached_meta) and os.path.exists(cached_info):
        print(f"[cache] {name} đã có kết quả — bỏ qua train lại")
        info = json.load(open(cached_info))
        meta = json.load(open(cached_meta))
    else:
        cmd = ["python", "experiments/baselines/finetune_pubmedbert.py",
               "--model", model, "--epochs", str(ep), "--lr", lr,
               "--batch-size", str(bs), "--max-len", str(ml), "--freeze-layers", str(fz)]
        try:
            subprocess.run(cmd, check=True)
        except subprocess.CalledProcessError as e:
            print(f"!! {name} LỖI ({e}) — bỏ qua, chạy config tiếp theo")
            continue
        info = json.load(open(CKPT_INFO))
        meta = json.load(open(META))
        os.makedirs(dest, exist_ok=True)
        shutil.copy(META, dest)
        shutil.copy(CKPT_INFO, dest)
        for png in CHARTS:
            shutil.copy(f"experiments/results/{png}", dest)
        # lưu trọng số riêng cho config này để resume qua phiên vẫn lấy lại được
        shutil.rmtree(cached_model, ignore_errors=True)
        shutil.copytree(MODEL_DIR, cached_model)
    rows.append({"config": name, "best_epoch": info["epoch"],
                 "val_roc_auc": round(info["val_roc_auc"], 4),
                 "val_pr_auc": round(info["val_pr_auc"], 4),
                 "test_recall": round(meta["metrics_test"]["recall"], 4),
                 "test_f1": round(meta["metrics_test"]["f1"], 4),
                 "test_roc_auc": round(meta["metrics_test"]["roc_auc"], 4),
                 "test_pr_auc": round(meta["metrics_test"]["pr_auc"], 4),
                 "test_wss95": round(meta["wss_test"], 4)})
    if info["val_roc_auc"] > best_val:
        best_val = info["val_roc_auc"]
        shutil.rmtree("sweep_results/best_model", ignore_errors=True)
        shutil.copytree(cached_model if os.path.exists(cached_model) else MODEL_DIR,
                        "sweep_results/best_model")

df = pd.DataFrame(rows).sort_values("val_roc_auc", ascending=False).reset_index(drop=True)
df.to_csv("sweep_results/sweep_summary.csv", index=False)
WINNER = df.iloc[0]["config"]
print("\n===== BẢNG TỔNG HỢP (xếp theo VAL ROC-AUC — cột test chỉ để tham khảo) =====")
print(df.to_string(index=False))
print(f"\n>>> Config thắng (theo validation): {WINNER}")

## 2.2 — Kiểm tra ổn định: 3 seeds trên config thắng

Cùng config, cùng tập test, chỉ đổi seed khởi tạo trọng số. Mean ± std cho biết con số có vững không hay chỉ là may mắn của một seed.

In [ ]:
import numpy as np

model, ep, lr, bs, ml, fz = CONFIGS[WINNER]
stab = [json.load(open(f"sweep_results/{WINNER}/finetuned_meta.json"))]  # seed 42 đã chạy ở sweep
for s in [123, 2026]:
    print(f"\n===== init-seed {s} =====")
    subprocess.run(["python", "experiments/baselines/finetune_pubmedbert.py",
                    "--model", model, "--epochs", str(ep), "--lr", lr,
                    "--batch-size", str(bs), "--max-len", str(ml),
                    "--freeze-layers", str(fz), "--init-seed", str(s)], check=True)
    stab.append(json.load(open(META)))

print(f"\n===== ỔN ĐỊNH QUA 3 SEEDS — {WINNER} (mean ± std, %) =====")
summary = {}
for k in ["precision", "recall", "f1", "roc_auc", "pr_auc"]:
    vals = [m["metrics_test"][k] for m in stab]
    summary[k] = (np.mean(vals) * 100, np.std(vals) * 100)
    print(f"  {k:10s}: {summary[k][0]:.1f} ± {summary[k][1]:.1f}")
wss_vals = [m["wss_test"] for m in stab]
print(f"  {'wss@95':10s}: {np.mean(wss_vals)*100:.1f} ± {np.std(wss_vals)*100:.1f}")
json.dump({k: {"mean": v[0], "std": v[1]} for k, v in summary.items()},
          open("sweep_results/stability_3seeds.json", "w"), indent=2)

## 2.3 — Đóng gói model thắng để tải về

Gói gồm: trọng số model thắng (seed 42), meta + 4 biểu đồ của nó, bảng sweep đầy đủ và kết quả ổn định 3 seeds.

In [ ]:
shutil.rmtree("deploy_bundle", ignore_errors=True)
os.makedirs("deploy_bundle/sr_core/screening_model", exist_ok=True)
os.makedirs("deploy_bundle/experiments/results", exist_ok=True)
shutil.copytree("sweep_results/best_model",
                "deploy_bundle/sr_core/screening_model/finetuned_pubmedbert")
shutil.copy(f"sweep_results/{WINNER}/finetuned_meta.json",
            "deploy_bundle/sr_core/screening_model/")
for png in CHARTS:
    shutil.copy(f"sweep_results/{WINNER}/{png}", "deploy_bundle/experiments/results/")
shutil.copy("sweep_results/sweep_summary.csv", "deploy_bundle/")
shutil.copy("sweep_results/stability_3seeds.json", "deploy_bundle/")
shutil.make_archive("ket_qua_round2", "zip", "deploy_bundle")
print(f"Model thắng: {WINNER}")
from google.colab import files
files.download("ket_qua_round2.zip")

---
# Đưa kết quả về máy local

Gửi file zip cho Claude Code trên máy local để phân tích + deploy, hoặc tự làm:
1. Giải nén, chép `finetuned_pubmedbert/` + `finetuned_meta.json` đè vào `sr_core/screening_model/`.
2. Chép 4 biểu đồ vào `experiments/results/`.
3. Số Bảng 5 = metrics trong `finetuned_meta.json`; số mean ± std trong `stability_3seeds.json` dùng cho phần bàn luận độ tin cậy.

**Chỉ thay model chính thức nếu config thắng vượt mốc hiện tại** (thí nghiệm 1A: test ROC-AUC 87.2%, recall 79.0%, F1 68.3%, WSS@95 33.4%). Lưu ý GPU không tái lập từng chữ số giữa các lần chạy — quyết định dựa trên chênh lệch rõ rệt (>0.5 điểm), không phải ±0.1.

---
# PHẦN 3 — Focal Loss + 5-seed ensemble (tối ưu recall)

**Mục tiêu:** đẩy recall từ 79% lên ≥ 88% và WSS@95 lên ≥ 40% một cách bền vững.

**Ba thay đổi so với vòng 2** (giữ nguyên config thắng lr=3e-5, freeze=2):

| Thay đổi | Lý do |
|---|---|
| **Focal Loss** γ=2, α=n_neg/total | Phạt nặng hơn khi bỏ sót bài Include; đưa biên quyết định sát lớp thiểu số hơn |
| **Early stopping trên val_PR-AUC** | PR-AUC nhạy hơn ROC-AUC với recall/precision tradeoff trên dữ liệu mất cân bằng |
| **Cosine LR + 16 epochs** | Ổn định hơn linear decay khi train dài; tìm được minimum tốt hơn |

**Quy trình:**
1. **3.1** — Chạy 1 lần (seed 42) xem cải thiện so với vòng 2 (~12 phút)
2. **3.2** — 5 seeds đo stability thực sự (~60 phút tổng; có cache, rớt Colab không mất)
3. **3.3** — Ensemble: trung bình xác suất 5 model → số ổn định nhất
4. **3.4** — Đóng gói tải về


## 3.1 — Chạy thử Focal Loss (seed 42, ~12 phút)

In [9]:
import os, json, shutil, subprocess

os.makedirs('r3_results/seed_42', exist_ok=True)

subprocess.run([
    'python', 'experiments/baselines/finetune_pubmedbert.py',
    '--model', 'nlpie/distil-biobert', '--epochs', '16', '--lr', '3e-5',
    '--batch-size', '16', '--max-len', '256', '--freeze-layers', '2',
    '--focal-loss', '--focal-gamma', '2.0',
    '--early-stop-on', 'pr_auc',
    '--lr-schedule', 'cosine',
    '--save-probs-to', 'r3_results/seed_42/test_probs.npy'
], check=True)

shutil.copy('sr_core/screening_model/finetuned_meta.json', 'r3_results/seed_42/')
shutil.copytree('sr_core/screening_model/finetuned_pubmedbert',
                'r3_results/seed_42/model', dirs_exist_ok=True)

m = json.load(open('r3_results/seed_42/finetuned_meta.json'))
r2 = {'precision':0.601,'recall':0.790,'f1':0.683,'roc_auc':0.872,'pr_auc':0.696}
print('\n===== KET QUA SEED 42 vs VONG 2 =====')
for k in ['precision','recall','f1','roc_auc','pr_auc']:
    v = m['metrics_test'][k]
    diff = (v - r2[k]) * 100
    arrow = 'UP' if diff > 0.1 else ('DOWN' if diff < -0.1 else '=')
    print(f'  {k:10s}: {v*100:.1f}%  ({arrow} {abs(diff):.1f} vs vong 2)')
print(f"  WSS@95    : {m['wss_test']*100:.1f}%  (vong 2: 33.4%)")



===== KET QUA SEED 42 vs VONG 2 =====
  precision : 58.4%  (DOWN 1.7 vs vong 2)
  recall    : 81.3%  (UP 2.3 vs vong 2)
  f1        : 68.0%  (DOWN 0.3 vs vong 2)
  roc_auc   : 87.9%  (UP 0.7 vs vong 2)
  pr_auc    : 70.1%  (UP 0.5 vs vong 2)
  WSS@95    : 36.7%  (vong 2: 33.4%)


## 3.2 — Stability 5 seeds (~60 phút, có cache)

Seed 42 đã có từ 3.1 → chạy thêm 4 seeds. Nếu Colab rớt, chạy lại cell — các seed đã xong sẽ bị bỏ qua.

In [10]:
import subprocess, json, shutil, os, numpy as np

META   = 'sr_core/screening_model/finetuned_meta.json'
R3     = 'r3_results'
SEEDS  = [42, 123, 2026, 777, 1234]
CMD    = ['python', 'experiments/baselines/finetune_pubmedbert.py',
          '--model', 'nlpie/distil-biobert', '--epochs', '16',
          '--lr', '3e-5', '--batch-size', '16',
          '--max-len', '256', '--freeze-layers', '2',
          '--focal-loss', '--focal-gamma', '2.0',
          '--early-stop-on', 'pr_auc', '--lr-schedule', 'cosine']

all_meta = {}
for s in SEEDS:
    dest   = f'{R3}/seed_{s}'
    cached = f'{dest}/finetuned_meta.json'
    if os.path.exists(cached):
        print(f'[cache] seed {s} đã có — bỏ qua')
        all_meta[s] = json.load(open(cached))
        continue
    print(f'\n{"="*20} seed {s} {"="*20}')
    os.makedirs(dest, exist_ok=True)
    probs_path = f'{dest}/test_probs.npy'
    subprocess.run(CMD + ['--init-seed', str(s), '--save-probs-to', probs_path], check=True)
    shutil.copy(META, dest)
    shutil.copytree('sr_core/screening_model/finetuned_pubmedbert',
                    f'{dest}/model', dirs_exist_ok=True)
    all_meta[s] = json.load(open(cached))

print('\n===== BẢNG 5 SEEDS =====')
print(f'{"seed":>6}  {"recall":>8}  {"f1":>8}  {"roc_auc":>8}  {"pr_auc":>8}  {"wss95":>8}')
for s in SEEDS:
    mm = all_meta[s]['metrics_test']; w = all_meta[s]['wss_test']
    print(f'{s:>6}  {mm["recall"]*100:>7.1f}%  {mm["f1"]*100:>7.1f}%  '
          f'{mm["roc_auc"]*100:>7.1f}%  {mm["pr_auc"]*100:>7.1f}%  {w*100:>7.1f}%')

print('\n===== MEAN ± STD (%) =====')
stab = {}
for k in ['precision','recall','f1','roc_auc','pr_auc']:
    vals = [all_meta[s]['metrics_test'][k] for s in SEEDS]
    stab[k] = {'mean': float(np.mean(vals)*100), 'std': float(np.std(vals)*100)}
    print(f'  {k:10s}: {stab[k]["mean"]:.1f} ± {stab[k]["std"]:.1f}')
wss_vals = [all_meta[s]['wss_test'] for s in SEEDS]
stab['wss95'] = {'mean': float(np.mean(wss_vals)*100), 'std': float(np.std(wss_vals)*100)}
print(f'  {"wss@95":10s}: {stab["wss95"]["mean"]:.1f} ± {stab["wss95"]["std"]:.1f}')
json.dump(stab, open(f'{R3}/stability_5seeds.json', 'w'), indent=2)


[cache] seed 42 đã có — bỏ qua

==================== seed 123 ====================

==================== seed 2026 ====================

==================== seed 777 ====================

==================== seed 1234 ====================

===== BẢNG 5 SEEDS =====
  seed    recall        f1   roc_auc    pr_auc     wss95
    42     81.3%     68.0%     87.9%     70.1%     36.7%
   123     70.3%     69.3%     87.2%     71.3%     29.1%
  2026     78.3%     69.7%     87.2%     69.5%     25.9%
   777     78.7%     68.1%     88.3%     73.4%     37.6%
  1234     81.7%     64.0%     86.7%     66.4%     32.7%

===== MEAN ± STD (%) =====
  precision : 60.4 ± 5.2
  recall    : 78.1 ± 4.1
  f1        : 67.8 ± 2.0
  roc_auc   : 87.4 ± 0.6
  pr_auc    : 70.2 ± 2.3
  wss@95    : 32.4 ± 4.5


## 3.3 — Ensemble: trung bình xác suất 5 model

Soft voting: trung bình xác suất → giảm variance, thường cải thiện thêm 0.5–1.5% so với model đơn.

In [11]:
import json, numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import (roc_auc_score, average_precision_score,
    precision_score, recall_score, f1_score)

R3    = 'r3_results'
SEEDS = [42, 123, 2026, 777, 1234]

# Định nghĩa inline (không import module để tránh lỗi path)
def load_labels():
    texts, labels = [], []
    with open('data/processed/labeled_dataset.jsonl', 'r', encoding='utf-8') as f:
        for line in f:
            line = line.strip()
            if not line: continue
            p = json.loads(line)
            title    = (p.get('title')    or '').strip()
            abstract = (p.get('abstract') or '').strip()
            texts.append(f'{title}. {abstract}'.strip())
            labels.append(1 if p.get('human_label') == 'include' else 0)
    return texts, np.array(labels)

def wss_at_recall(y_true, y_prob, target=0.95):
    from sklearn.metrics import confusion_matrix
    n = len(y_true)
    best = (0.0, 0.5)
    for t in np.unique(y_prob):
        pred = (y_prob >= t).astype(int)
        tn, fp, fn, tp = confusion_matrix(y_true, pred, labels=[0,1]).ravel()
        rec = tp / (tp + fn) if (tp + fn) else 0.0
        if rec >= target:
            wss = (tn + fn) / n - (1 - target)
            if wss > best[0]: best = (wss, float(t))
    return best[0]

# Lấy test labels từ split cố định (seed=42)
texts, y = load_labels()
X_tmp, X_te, y_tmp, y_te = train_test_split(texts, y, test_size=0.15,
                                             stratify=y, random_state=42)
_, _, y_tr, y_va_te = train_test_split(X_tmp, y_tmp, test_size=0.1765,
                                       stratify=y_tmp, random_state=42)
print(f'Test set: {len(y_te)} samples, {int(y_te.sum())} include')

# Load và average xac suat tu 5 seeds
prob_arrays = [np.load(f'{R3}/seed_{s}/test_probs.npy') for s in SEEDS]
pt_ens = np.mean(prob_arrays, axis=0)

# Threshold toi uu F1
best_t, best_f1 = 0.5, -1.0
for t in np.linspace(0.05, 0.95, 91):
    f1 = f1_score(y_te, (pt_ens >= t).astype(int), zero_division=0)
    if f1 > best_f1: best_f1, best_t = f1, float(t)

pred  = (pt_ens >= best_t).astype(int)
m_ens = {
    'precision': precision_score(y_te, pred, zero_division=0),
    'recall':    recall_score(y_te, pred, zero_division=0),
    'f1':        f1_score(y_te, pred, zero_division=0),
    'roc_auc':   roc_auc_score(y_te, pt_ens),
    'pr_auc':    average_precision_score(y_te, pt_ens),
}
wss_ens = wss_at_recall(y_te, pt_ens)

r2 = {'precision':0.601,'recall':0.790,'f1':0.683,'roc_auc':0.872,'pr_auc':0.696}
print('\n===== ENSEMBLE (5 seeds) vs VONG 2 =====')
for k in ['precision','recall','f1','roc_auc','pr_auc']:
    diff = (m_ens[k] - r2[k]) * 100
    arrow = 'UP' if diff > 0.1 else ('DOWN' if diff < -0.1 else '=')
    print(f'  {k:10s}: {m_ens[k]*100:.1f}%  ({arrow} {abs(diff):.1f})')
print(f"  WSS@95    : {wss_ens*100:.1f}%  (vong 2: 33.4%)")
print(f'  Threshold : {best_t:.2f}')
json.dump({**{k: float(v) for k,v in m_ens.items()},
           'wss95': float(wss_ens), 'threshold_f1': best_t, 'seeds': SEEDS},
          open(f'{R3}/ensemble_metrics.json', 'w'), indent=2)


Test set: 1201 samples, 300 include

===== ENSEMBLE (5 seeds) vs VONG 2 =====
  precision : 63.9%  (UP 3.8)
  recall    : 79.0%  (= 0.0)
  f1        : 70.6%  (UP 2.3)
  roc_auc   : 89.0%  (UP 1.8)
  pr_auc    : 72.2%  (UP 2.6)
  WSS@95    : 40.5%  (vong 2: 33.4%)
  Threshold : 0.44


## 3.4 — Đóng gói kết quả Round 3

In [12]:
import shutil, os, json, numpy as np
from google.colab import files

R3 = 'r3_results'
SEEDS = [42, 123, 2026, 777, 1234]
CHARTS = ['transformer_confusion_matrix.png','transformer_roc_curve.png',
          'transformer_pr_curve.png','transformer_probability_distribution.png']

# Chọn model đơn tốt nhất theo recall
best_seed = max(SEEDS, key=lambda s: json.load(
    open(f'{R3}/seed_{s}/finetuned_meta.json'))['metrics_test']['recall'])
print(f'Model đơn tốt nhất: seed {best_seed}')

shutil.rmtree('r3_bundle', ignore_errors=True)
os.makedirs('r3_bundle/sr_core/screening_model', exist_ok=True)
os.makedirs('r3_bundle/experiments/results', exist_ok=True)
os.makedirs('r3_bundle/r3_probs', exist_ok=True)

shutil.copytree(f'{R3}/seed_{best_seed}/model',
                'r3_bundle/sr_core/screening_model/finetuned_pubmedbert')
shutil.copy(f'{R3}/seed_{best_seed}/finetuned_meta.json',
            'r3_bundle/sr_core/screening_model/')
for png in CHARTS:
    src = f'experiments/results/{png}'
    if os.path.exists(src): shutil.copy(src, 'r3_bundle/experiments/results/')
shutil.copy(f'{R3}/stability_5seeds.json', 'r3_bundle/')
shutil.copy(f'{R3}/ensemble_metrics.json', 'r3_bundle/')
for s in SEEDS:
    src = f'{R3}/seed_{s}/test_probs.npy'
    if os.path.exists(src): shutil.copy(src, f'r3_bundle/r3_probs/seed_{s}_probs.npy')

shutil.make_archive('ket_qua_round3', 'zip', 'r3_bundle')
print('Đóng gói xong → ket_qua_round3.zip')
files.download('ket_qua_round3.zip')


Model đơn tốt nhất: seed 1234
Đóng gói xong → ket_qua_round3.zip


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

---
# Đưa kết quả Round 3 về máy local

Gửi `ket_qua_round3.zip` cho Claude Code để phân tích + deploy:
1. Giải nén → chép `finetuned_pubmedbert/` + `finetuned_meta.json` vào `sr_core/screening_model/`
2. Chép 4 biểu đồ vào `experiments/results/`
3. **Bảng 5 luận văn** = số trong `stability_5seeds.json` (mean ± std, 5 seeds)
4. **Ensemble** = số trong `ensemble_metrics.json` → đưa vào phần thảo luận

**Thay model chính thức nếu recall ≥ 85% VÀ WSS@95 ≥ 40%** (ngưỡng đáng kể so với vòng 2: 79%, 33.4%).